# PrometheusNet training on PUMA

Colab Pro entry point for the instance-aware multitask architecture. Tissue is trained as semantic segmentation; nuclei are trained as center-based instances and evaluated with centroid F1.

## 1. Environment

In [ ]:
import importlib
import os
import sys
from pathlib import Path

REPO_DIR = '/content/prometheus'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone https://github.com/hoangtung386/Prometheus.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

In [ ]:
%cd {REPO_DIR}
%pip install -q -r requirements.txt

In [ ]:
# Prefer this checkout over any unrelated third-party package named `prometheus`.
SOURCE_DIR = str(Path(REPO_DIR, 'src').resolve())
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)

sys.path.insert(0, SOURCE_DIR)
loaded = sys.modules.get('prometheus')
loaded_file = getattr(loaded, '__file__', None) if loaded is not None else None
if loaded is not None and (loaded_file is None or SOURCE_DIR not in str(Path(loaded_file).resolve())):
    for module_name in [name for name in sys.modules if name == 'prometheus' or name.startswith('prometheus.')]:
        del sys.modules[module_name]

In [ ]:
importlib.invalidate_caches()
from prometheus import api as prometheus_api

print('Prometheus API:', Path(prometheus_api.__file__).resolve())
!git rev-parse --short HEAD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU is required. In Colab select Runtime > Change runtime type > GPU, then rerun.')

print(f'PyTorch {torch.__version__}; CUDA {torch.version.cuda}')
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, torch.cuda.get_device_name(index), f'{props.total_memory / 2**30:.1f} GiB')

## 2. Resolved configuration

In [ ]:
from dataclasses import asdict
from pprint import pprint
from pathlib import Path
from prometheus.api import build_criterion, build_datamodule, build_model, build_trainer, load_config

CONFIG_PATH = 'configs/experiment/baseline_multitask.toml'
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/PUMA/dataset_PUMA')
RUN_DIR = Path('/content/drive/MyDrive/prometheus_runs/baseline_multitask_v1')

if not DRIVE_DATASET_DIR.is_dir():
    raise FileNotFoundError(f'PUMA dataset not found: {DRIVE_DATASET_DIR}')

RUN_DIR.mkdir(parents=True, exist_ok=True)
config = load_config(CONFIG_PATH)
config.data.root = str(DRIVE_DATASET_DIR)
config.data.split_manifest = str(RUN_DIR / 'puma_split.json')
config.paths.run_dir = str(RUN_DIR)

# --- Hardware overrides for a single large-VRAM GPU (A6000 48GB / A100 / H100) ---
# The dataset is small (~205 pairs -> ~185 train / ~20 val at validation_fraction=0.1),
# so the binding constraint is the NUMBER OF OPTIMIZER STEPS PER EPOCH, not VRAM.
# steps/epoch = ceil(185 / batch_size): batch 8 -> ~24, batch 16 -> ~12, batch 32 -> ~6.
# We use a real batch of 16 (no gradient accumulation): it fills most of the GPU at
# 1024px while keeping ~12 steps/epoch. Push to 24-32 only if metrics are still healthy.
config.trainer.batch_size = 16
config.trainer.gradient_accumulation = 1  # not needed on a big GPU; real batch = effective batch
config.trainer.num_workers = 8            # match to available CPU cores (drop to 4 on Colab Pro)
config.trainer.amp = True                 # mixed precision: less VRAM + faster, keep on

# The trainer applies NO automatic LR scaling and its cosine schedule steps PER EPOCH,
# so the LR must be scaled by hand for the larger effective batch. The 2e-4 baseline was
# tuned for effective batch 4; sqrt-scaling to batch 16 -> 2e-4 * sqrt(16/4) = 4e-4.
config.optimizer.lr = 4e-4
config.trainer.min_lr = 1e-6              # must stay <= optimizer.lr

config.validate()

effective_batch = config.trainer.batch_size * config.trainer.gradient_accumulation
print(f'effective_batch={effective_batch}  lr={config.optimizer.lr}  workers={config.trainer.num_workers}')
pprint(asdict(config))

## 3. Data and one-batch smoke test

In [ ]:
from prometheus.data.puma.audit import audit_puma_dataset

audit = audit_puma_dataset(config.data.root)
print('samples', audit['sample_count'])
if audit['errors']:
    raise ValueError(f"Dataset audit failed: {audit['errors'][:5]}")

train_loader, validation_loader = build_datamodule(config)
batch = next(iter(train_loader))

print('images', tuple(batch.images.shape))
print('tissue', tuple(batch.tissue.mask.shape))
print('instances/image', [len(target.labels) for target in batch.nuclei])

### Visual inspection after normalization

The preview rescales the normalized tensor only for display. The model still receives the normalized tensor unchanged.

In [ ]:
from prometheus.visualization import visualize_multitask_batch

visualize_multitask_batch(batch, index=0, show_boxes=True)

In [ ]:
import gc

device = torch.device('cuda')
torch.cuda.reset_peak_memory_stats(device)

model = build_model(config).to(device)
batch = batch.to(device)
criterion = build_criterion(config)

with torch.amp.autocast('cuda', enabled=config.trainer.amp):
    output = model(batch.images)
    losses = criterion(output, batch)

losses['total'].backward()

print({name: round(value.detach().item(), 5) for name, value in losses.items()})

# VRAM probe: this ran a real fwd+bwd on the configured batch size, so peak here ~= the
# per-step training memory (training only adds a small, fixed AdamW optimizer state on top).
# Use the reported headroom to decide whether to raise config.trainer.batch_size above.
peak = torch.cuda.max_memory_allocated(device) / 2**30
total = torch.cuda.get_device_properties(device).total_memory / 2**30
print(f'batch_size={config.trainer.batch_size}  peak_vram={peak:.2f} GiB / {total:.1f} GiB '
      f'({100 * peak / total:.0f}% used, {total - peak:.1f} GiB free)')

del batch, output, losses, criterion, model

In [ ]:
gc.collect()
torch.cuda.empty_cache()

## 4. Train or resume

In [ ]:
model = build_model(config)
trainer = build_trainer(config, model=model, datamodule=(train_loader, validation_loader), device=device)
last_checkpoint = Path(config.paths.run_dir) / 'last.ckpt'
resume_from = last_checkpoint if last_checkpoint.exists() else None
metrics = trainer.fit(resume_from=resume_from)

print(metrics)

## 5. Exact validation and artifacts

In [ ]:
from prometheus.engine import evaluate_multitask

best_path = Path(config.paths.run_dir) / 'best_primary.ckpt'
trainer.resume(best_path)

result = evaluate_multitask(
    trainer.model, validation_loader, trainer.criterion, device,
    config.model.nuclei_feature_stride, config.evaluation.nuclei_radius_px,
    config.postprocess.confidence_threshold, config.postprocess.max_detections,
    config.postprocess.local_max_kernel,
)

print('tissue Dice:', result.tissue_dice)
print('nuclei centroid F1:', result.nuclei_macro_f1)
print('checkpoint:', best_path)


## 6. Submission smoke test

Use the same checkpoint through the CLI so notebook and headless inference share one composition path:

```bash
prometheus predict --config configs/experiment/baseline_multitask.toml --checkpoint /path/to/best_primary.ckpt --input /path/to/sample.tif --output /content/prediction
```